# Hugging Face Text Classification Workflow

## Notebook Contract
- **Objective:** provide a reproducible development workflow with offline baseline modeling and optional Hugging Face fine-tuning.
- **Inputs:** synthetic support-ticket data generated locally.
- **Outputs:** dataset preview, baseline metrics, error analysis tables, and model-card preview.
- **Expected runtime:** 3-5 minutes on CPU with default settings.
- **Scope boundary:** production logic stays in `src/hf_finetuning_lab`; notebook orchestrates and visualizes.


## 1) Setup and Reproducibility


In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path('..').resolve()
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from hf_finetuning_lab.data.io import build_label_mapping, validate_text_classification_frame
from hf_finetuning_lab.model_cards.model_card import write_model_card
from hf_finetuning_lab.sample_data import write_sample_data

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f"Timestamp (UTC): {datetime.now(UTC).isoformat(timespec='seconds')}")
print(f'Seed: {SEED}')


## 2) Parameters


In [ ]:
DATA_PATH = ROOT / 'data' / 'raw' / 'support_tickets.csv'
REPORTS_DIR = ROOT / 'reports' / 'sample_run'
MODEL_DIR = ROOT / 'artifacts' / 'models' / 'support-triage-notebook'
BASELINE_METRICS_PATH = REPORTS_DIR / 'baseline_metrics.json'
BASELINE_ERRORS_PATH = REPORTS_DIR / 'baseline_errors.csv'
BASELINE_MODEL_CARD_PATH = REPORTS_DIR / 'baseline_model_card.md'
HF_EVAL_PATH = REPORTS_DIR / 'evaluation.json'
HF_PRED_PATH = REPORTS_DIR / 'predictions.csv'

MODEL_NAME = 'distilbert-base-uncased'
TEXT_COL = 'text'
LABEL_COL = 'label'
ROWS = 800
TEST_SIZE = 0.2
MAX_FEATURES = 12000
EPOCHS = 1
BATCH_SIZE = 8
RUN_HF_COMMANDS = False

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Root: {ROOT}')
print(f'Data path: {DATA_PATH}')
print(f'Reports dir: {REPORTS_DIR}')


## 3) Data Generation, Loading, and Validation


In [ ]:
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
write_sample_data(output=DATA_PATH, rows=ROWS, seed=SEED)

df = pd.read_csv(DATA_PATH)
validate_text_classification_frame(df, text_col=TEXT_COL, label_col=LABEL_COL)

label2id, id2label = build_label_mapping(df[LABEL_COL])
print(f'Rows: {len(df)}')
print(f'Labels: {list(label2id.keys())}')
df.head()


## 4) Exploratory Data Analysis


In [ ]:
label_distribution = df[LABEL_COL].value_counts().rename_axis('label').reset_index(name='count')
label_distribution['share'] = (label_distribution['count'] / len(df)).round(4)
label_distribution


In [ ]:
df['text_len'] = df[TEXT_COL].str.len()
df['token_estimate'] = df[TEXT_COL].str.split().str.len()
df[['text_len', 'token_estimate']].describe().T


## 5) Offline Baseline Model (Scikit-learn)

This section is fully runnable without Hugging Face downloads and should be the default local debugging path.


In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df[LABEL_COL],
)

baseline_model = Pipeline(
    steps=[
        ('tfidf', TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2))),
        ('clf', LogisticRegression(max_iter=1000, multi_class='auto', random_state=SEED)),
    ]
)
baseline_model.fit(train_df[TEXT_COL], train_df[LABEL_COL])

pred_labels = baseline_model.predict(test_df[TEXT_COL])
macro_f1 = float(f1_score(test_df[LABEL_COL], pred_labels, average='macro'))
weighted_f1 = float(f1_score(test_df[LABEL_COL], pred_labels, average='weighted'))

print(f'Baseline macro F1: {macro_f1:.4f}')
print(f'Baseline weighted F1: {weighted_f1:.4f}')


## 6) Evaluation and Error Analysis


In [ ]:
report = classification_report(
    test_df[LABEL_COL],
    pred_labels,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
report_df


In [ ]:
labels_sorted = sorted(test_df[LABEL_COL].unique().tolist())
cm = confusion_matrix(test_df[LABEL_COL], pred_labels, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)
cm_df


In [ ]:
error_df = test_df[[TEXT_COL, LABEL_COL]].copy()
error_df['predicted_label'] = pred_labels
error_df['is_error'] = error_df[LABEL_COL] != error_df['predicted_label']

hard_examples = error_df[error_df['is_error']].head(20)
hard_examples


In [ ]:
baseline_metrics = {
    'macro_f1': macro_f1,
    'weighted_f1': weighted_f1,
    'test_rows': int(len(test_df)),
    'labels': labels_sorted,
}

BASELINE_METRICS_PATH.write_text(json.dumps(baseline_metrics, indent=2), encoding='utf-8')
error_df.to_csv(BASELINE_ERRORS_PATH, index=False)

print(f'Saved baseline metrics to: {BASELINE_METRICS_PATH}')
print(f'Saved baseline errors to: {BASELINE_ERRORS_PATH}')


## 7) Model Card Preview (Baseline)


In [ ]:
limitations = [
    'Dataset labels are synthetic and do not represent a real operational taxonomy.',
    'Metrics are from a synthetic dataset and are not a proxy for production performance.',
    'No clinical, legal, or safety validity is claimed by this workflow.',
]

card_path = write_model_card(
    output_path=BASELINE_MODEL_CARD_PATH,
    model_name='tfidf-logreg-baseline',
    task='text-classification',
    label_names=labels_sorted,
    metrics={'macro_f1': macro_f1, 'weighted_f1': weighted_f1},
    limitations=limitations,
)
print(f'Wrote baseline model card to: {card_path}')
print(card_path.read_text(encoding='utf-8')[:900])


## 8) Optional Hugging Face Training and Evaluation (CLI)

Set `RUN_HF_COMMANDS=True` only when you explicitly want model downloads/training.


In [ ]:
train_cmd = [
    'poetry', 'run', 'hf-lab', 'train',
    '--input', str(DATA_PATH),
    '--output-dir', str(MODEL_DIR),
    '--model-name', MODEL_NAME,
    '--text-col', TEXT_COL,
    '--label-col', LABEL_COL,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
]

eval_cmd = [
    'poetry', 'run', 'hf-lab', 'evaluate',
    '--model-dir', str(MODEL_DIR),
    '--input', str(DATA_PATH),
    '--output', str(HF_EVAL_PATH),
    '--text-col', TEXT_COL,
    '--label-col', LABEL_COL,
]

predict_cmd = [
    'poetry', 'run', 'hf-lab', 'predict',
    '--model-dir', str(MODEL_DIR),
    '--input', str(DATA_PATH),
    '--output', str(HF_PRED_PATH),
    '--text-col', TEXT_COL,
]

print('Train command:')
print(' '.join(train_cmd))
print('Evaluate command:')
print(' '.join(eval_cmd))
print('Predict command:')
print(' '.join(predict_cmd))

if RUN_HF_COMMANDS:
    subprocess.run(train_cmd, check=True, cwd=ROOT)
    subprocess.run(eval_cmd, check=True, cwd=ROOT)
    subprocess.run(predict_cmd, check=True, cwd=ROOT)
    print('HF CLI pipeline executed successfully.')
else:
    print('HF CLI pipeline skipped (RUN_HF_COMMANDS=False).')


## 9) Checklist
- Baseline metrics and error table generated under `reports/sample_run/`.
- Model card includes synthetic-label limitations and non-validity disclaimers.
- HF workflow command strings are ready for opt-in execution.
- For production: use real data validation, subgroup analysis, and failure-mode review.
